# IVModel Mechanistic Interpretability

Visualizes what the trained `IVModel` checkpoints learned: attention patterns, QK circuits, and fixed-lag attention detection (a numeric-time-series analog of GPT-2's induction heads). Runs on both `Pratham007xo/iv-forecast-constituent-124m` and `Pratham007xo/iv-forecast-index-124m`.

## How to read these results

This is a genuinely trained ~124M-parameter causal decoder, so what it actually learned is an
empirical question -- the cells below don't assume a particular answer, they just make it
measurable. A few domain-grounded hypotheses worth checking against the real output (not
predictions of what you'll see, just what to look for):

- **Recency bias**: IV is highly autocorrelated day-to-day, so a plausible finding is that early
  layers have heads strongly attending to lag 1 (yesterday) -- a simple "persistence" signal --
  while later layers integrate longer-range context (lag 5, ~1 week; lag 21, ~1 month).
- **Diffuse vs. peaked attention**: if `fixed_lag_detection` flags few or no heads as fixed-lag,
  that's a real (and informative) finding too -- it would suggest the model relies more on the
  *content* of recent days (the actual feature values) than on a fixed positional offset.
- **Constituent vs. index model differences**: the index-level model is trained on a much
  smoother, equal-weighted-average series (less idiosyncratic noise per ticker) with far fewer
  training windows than the constituent model. It's plausible the two develop different
  attention structure purely from that data-distribution difference, independent of "skill."

Every claim below the `## Interpretation` heading at the end of this notebook is generated
directly from the DataFrames/dicts computed above it -- it reflects whatever this run actually
found, not a scripted narrative.

## Setup

In [ ]:
!git clone https://github.com/prathamkul007-max/IV_mech_interp_model.git
%cd IV_mech_interp_model
!pip install -q -r requirements.txt
!pip install -q -r Mechanistic-Interpretability/requirements.txt

In [ ]:
import sys
sys.path.insert(0, 'Mechanistic-Interpretability')

import torch
print('CUDA available:', torch.cuda.is_available())

## Prepare validation data (needed as real model input for the visualizations)

In [ ]:
!python scripts/prepare_options_iv.py --output-dir options_iv_data --seq-length 32 --stride 5
!python scripts/prepare_options_iv_index.py --output-dir options_iv_index_data --seq-length 32 --stride 1

## Load both checkpoints from the Hugging Face Hub

In [ ]:
from load_iv_model import load_iv_checkpoint

constituent_model, constituent_scaler = load_iv_checkpoint('Pratham007xo/iv-forecast-constituent-124m')
index_model, index_scaler = load_iv_checkpoint('Pratham007xo/iv-forecast-index-124m')

print('Constituent model:', constituent_model.config)
print('Index model:', index_model.config)

## Verify hooks against both real checkpoints

In [ ]:
from hooks import verify_hooks
from sample_windows import load_sample_windows

constituent_windows = load_sample_windows('options_iv_data/valid.npz', n=200)
index_windows = load_sample_windows('options_iv_index_data/valid.npz', n=200)

constituent_features = torch.from_numpy(constituent_windows).float()
index_features = torch.from_numpy(index_windows).float()

constituent_hook_results = verify_hooks(constituent_model, constituent_features[:8])
index_hook_results = verify_hooks(index_model, index_features[:8])

print('Constituent model hook verification:', constituent_hook_results)
print('Index model hook verification:', index_hook_results)

assert constituent_hook_results['all_passed'], 'Hooks failed verification on the constituent model -- do not trust downstream visualizations'
assert index_hook_results['all_passed'], 'Hooks failed verification on the index model -- do not trust downstream visualizations'

## Attention pattern visualization

In [ ]:
from visualize_attention import plot_attention_heatmap_static, plot_attention_heatmap_interactive, plot_all_heads_grid

plot_attention_heatmap_static(constituent_model, constituent_features[:1], layer=0, head=0)
plot_all_heads_grid(constituent_model, constituent_features[:1], layer=constituent_model.config.num_layers - 1)

In [ ]:
fig = plot_attention_heatmap_interactive(constituent_model, constituent_features[:1], layer=0, head=0)
fig.show()

## QK circuit analysis

In [ ]:
from qk_circuit_analysis import plot_qk_interactions_static, plot_qk_interactions_interactive

plot_qk_interactions_static(constituent_model, constituent_features[:8], layer=0)
fig = plot_qk_interactions_interactive(constituent_model, constituent_features[:8], layer=constituent_model.config.num_layers - 1)
fig.show()

## Fixed-lag attention detection

In [ ]:
from fixed_lag_detection import compute_lag_profile, plot_lag_heatmap, plot_layer_scores

constituent_lag_df = compute_lag_profile(constituent_model, constituent_features)
constituent_lag_df.sort_values('peak_score', ascending=False).head(10)

In [ ]:
plot_lag_heatmap(constituent_lag_df)
plot_layer_scores(constituent_lag_df, layer=0)

## Repeat all three analyses for the index-level model

In [ ]:
plot_attention_heatmap_static(index_model, index_features[:1], layer=0, head=0)
plot_qk_interactions_static(index_model, index_features[:8], layer=0)

index_lag_df = compute_lag_profile(index_model, index_features)
plot_lag_heatmap(index_lag_df)
index_lag_df.sort_values('peak_score', ascending=False).head(10)

## Interpretation

Auto-generated from the actual `constituent_lag_df` / `index_lag_df` / hook-verification results
computed above -- not a scripted narrative. Re-running this notebook (e.g. after retraining, or
with different sample windows) will regenerate different text if the underlying numbers differ.

In [ ]:
from IPython.display import Markdown, display
from interpret import summarize_lag_profile, compare_lag_profiles, summarize_hook_verification

report = '\n\n'.join([
    summarize_hook_verification(constituent_hook_results, 'Constituent-level model'),
    summarize_hook_verification(index_hook_results, 'Index-level model'),
    summarize_lag_profile(constituent_lag_df, 'Constituent-level model'),
    summarize_lag_profile(index_lag_df, 'Index-level model'),
    compare_lag_profiles(constituent_lag_df, index_lag_df),
])

display(Markdown(report))

with open('interpretation_report.md', 'w') as f:
    f.write(report)
print('\nSaved to interpretation_report.md')